# Pixal3D official native baseline

**Unvalidated build/inference recipe, not historical T4 acceptance.** Select an SM80/86/89 GPU explicitly (for example L4). Free Colab T4 is rejected; no automatic GPU upgrade or legacy fallback. Native source builds and roughly 25–30 GB of weights need substantial time, RAM and disk. Model licenses/access, including RMBG-2.0, remain your responsibility. This notebook runs compute only when you execute its cells.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

hardware = json.loads(subprocess.check_output([sys.executable, "-c", "import json,torch; assert torch.cuda.is_available(); print(json.dumps(torch.cuda.get_device_capability()))"], text=True))
CUDA_ARCH = ".".join(str(value) for value in hardware)
if CUDA_ARCH not in ("8.0", "8.6", "8.9"):
    raise RuntimeError("Select an SM80/86/89 GPU explicitly. T4 is not the official BF16 baseline.")
ROOT = Path("/content/pixal3d-cuda-native")
SOURCE_REF = "main"
if not ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", SOURCE_REF, "https://github.com/syi0808/pixal3d-cuda.git", str(ROOT)])
os.chdir(ROOT)
print("Wrapper revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
run_command([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run_command([sys.executable, "-m", "pip", "install", "-e", ".[upstream,camera]"])
run_command([sys.executable, "-m", "pixalcuda.upstream_sources", "--build", "--cuda-arch", CUDA_ARCH])
run_command([sys.executable, "-m", "pixalcuda.cli", "doctor"])


## Prefetch pinned weights
Store a Hugging Face read token in Colab Secrets as `HF_TOKEN`; do not paste it into the notebook or logs. The worker runs offline after prefetch. Upstream and native dependency source revisions are pinned in `upstream-lock.json`; wrapper HEAD is printed and hashed in the run report.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
run_command([sys.executable, "-m", "pixalcuda.cli", "download"])


## Upload one image and generate
Default: official low-VRAM / SDPA, 1024 geometry, dedicated texture NAF 1024, 4096 PBR textures, original CUDA remesh/export. This is not a speed/fit guarantee. Use a fresh output directory. Native log remains available if inference fails.

In [ ]:
from google.colab import files
import time

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one image.")
IMAGE = str(Path(next(iter(uploaded))).resolve())
OUTPUT = ROOT / "outputs" / f"native-{int(time.time())}"
print("Output:", OUTPUT)
run_command([sys.executable, "-m", "pixalcuda.cli", "generate", "--image", IMAGE, "--output-dir", str(OUTPUT), "--save-reference"])


## Inspect report and download
`reference.pt` contains private input-derived native latents and raw mesh/voxel tensors. It is for parity testing, not public upload. The report/GLB validator checks embedded PBR structure, not visual quality or watertightness.

In [ ]:
report = json.loads((OUTPUT / "report.json").read_text())
print(json.dumps({"status": report["status"], "gpu": report.get("gpu"), "actual_geometry_resolution": report["actual_geometry_resolution"], "conditioning": report["conditioning"], "stages": report["stages"], "validation": report["validation"]}, indent=2))
files.download(str(OUTPUT / "result.glb"))
